In [2]:
# 모듈 import
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.utils import plot_model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
from sklearn.preprocessing import LabelEncoder

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU, compute capability 8.6


In [3]:
# 이미지 갖고오기
df = pd.read_csv('data/cifar10/trainLabels.csv')
image_dir= './data/cifar10/train/'
# x데이터랑 t데이터 매칭
x_data = [image_dir + str(fname) + '.png' for fname in df['id']]
# t데이터 숫자로 인코딩
le = LabelEncoder()
t_data = le.fit_transform(df['label'])

In [4]:
# 데이터 확인
print(x_data[:5])
print(t_data[:7])

['./data/cifar10/train/1.png', './data/cifar10/train/2.png', './data/cifar10/train/3.png', './data/cifar10/train/4.png', './data/cifar10/train/5.png']
[6 9 9 4 1 1 2]


In [5]:
# Parameter 설정
IMAGE_SIZE = 224
BATCH_SIZE = 32

In [6]:
# ImageDataGenerator 생성
df_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                               validation_split=0.2)

In [7]:
df['id'] = df['id'].astype(str)+ '.png'
print(df['id'].head())

0    1.png
1    2.png
2    3.png
3    4.png
4    5.png
Name: id, dtype: object


In [10]:
# ImageDataGenerator 설정에서 subset을 이용해 train/validation 분리하기
train_generator = df_datagen.flow_from_dataframe(
    dataframe=df,
    directory='./data/cifar10/train/train',
    x_col='id',
    y_col='label',
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'  # 훈련 데이터
)

validation_generator = df_datagen.flow_from_dataframe(
    dataframe=df,
    directory='./data/cifar10/train/train',
    x_col='id',
    y_col='label',
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'  # 검증 데이터
)

Found 40000 validated image filenames belonging to 10 classes.
Found 10000 validated image filenames belonging to 10 classes.


In [11]:
# model
model_base = EfficientNetB0(weights='imagenet',
                            include_top=False,
                            input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
for layer in model_base.layers:
    layer.trainable = False

In [12]:
model = Sequential()
model.add(model_base)
model.add(GlobalAveragePooling2D())
model.add(Dense(units=64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(units=10,
               activation='softmax'))

In [13]:
# model 설정
model.compile(optimizer=Adam(learning_rate=1e-4),
             loss='categorical_crossentropy',
             metrics=['accuracy'])

In [14]:
es_callback = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True,
                           verbose=1)
cp_callback = ModelCheckpoint(filepath='./efficientnetb0_weights.h5',
                             save_best_only=True,
                             save_weights_only=True,
                             monitor='val_accuracy',
                             verbose=1)

In [15]:
# model 1차 학습
model.fit(train_generator,
         steps_per_epoch=len(train_generator),
         epochs=20,
         validation_data=validation_generator,
         validation_steps=len(validation_generator),
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/20
1249/1250 [============================>.] - ETA: 0s - loss: 1.6473 - accuracy: 0.4288     
Epoch 1: val_accuracy improved from -inf to 0.61720, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 59s 43ms/step - loss: 1.6469 - accuracy: 0.4290 - val_loss: 1.1621 - val_accuracy: 0.6172
Epoch 2/20
1249/1250 [============================>.] - ETA: 0s - loss: 1.2576 - accuracy: 0.5704  
Epoch 2: val_accuracy improved from 0.61720 to 0.66890, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 53s 43ms/step - loss: 1.2577 - accuracy: 0.5703 - val_loss: 0.9887 - val_accuracy: 0.6689
Epoch 3/20
1249/1250 [============================>.] - ETA: 0s - loss: 1.1271 - accuracy: 0.6142  
Epoch 3: val_accuracy improved from 0.66890 to 0.69270, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 53s 42ms/step - loss: 1.1271 - accuracy: 0.6143 - val_loss: 0.9075 - val_accuracy: 0.

In [16]:
model_base.summary()

Model: "efficientnetb0"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 rescaling_2 (Rescaling)     (None, 224, 224, 3)          0         ['input_2[0][0]']             
                                                                                                  
 normalization_1 (Normaliza  (None, 224, 224, 3)          7         ['rescaling_2[0][0]']         
 tion)                                                                                            
                                                                                                  
 rescaling_3 (Rescaling)     (None, 224, 224, 3)          0         ['normalization_1

In [17]:
# Fine Tuning
model_base.trainable = True

for layer in model_base.layers[:-30]:
    layer.trainable = False

In [18]:
# model 재설정
model.compile(optimizer=Adam(learning_rate=1e-4),
             loss='categorical_crossentropy',
             metrics=['accuracy'])

In [20]:
# model 재학습
model.fit(train_generator,
         steps_per_epoch=len(train_generator),
         epochs=20,
         validation_data=validation_generator,
         validation_steps=len(validation_generator),
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/20
1250/1250 [==============================] - ETA: 0s - loss: 0.9257 - accuracy: 0.6883     
Epoch 1: val_accuracy improved from 0.76580 to 0.82050, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 70s 52ms/step - loss: 0.9257 - accuracy: 0.6883 - val_loss: 0.5247 - val_accuracy: 0.8205
Epoch 2/20
1249/1250 [============================>.] - ETA: 0s - loss: 0.6443 - accuracy: 0.7805  
Epoch 2: val_accuracy improved from 0.82050 to 0.84600, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 64s 51ms/step - loss: 0.6444 - accuracy: 0.7805 - val_loss: 0.4407 - val_accuracy: 0.8460
Epoch 3/20
1249/1250 [============================>.] - ETA: 0s - loss: 0.5542 - accuracy: 0.8123  
Epoch 3: val_accuracy improved from 0.84600 to 0.86100, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 64s 51ms/step - loss: 0.5539 - accuracy: 0.8124 - val_loss: 0.3973 - val_accuracy:

In [24]:
# 테스트 제출 파일 만들기
# 1. submission 파일 먼저 불러오기 (test 이미지 id가 들어 있음)
submission = pd.read_csv('./data/cifar10/sampleSubmission.csv')  # 총 300,000행

submission['id'] = submission['id'].astype(str)+ '.png'  # 정수 → 문자열
print(submission['id'].head())

0    1.png
1    2.png
2    3.png
3    4.png
4    5.png
Name: id, dtype: object


In [25]:
# 2. test 데이터 제너레이터 만들기
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=submission,
    directory='data/cifar10/test/test',
    x_col='id',
    y_col=None,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    class_mode=None,
    shuffle=False,  # 반드시 순서 유지해야 함
    batch_size=32
)

Found 300000 validated image filenames.


In [26]:
# 3. 예측 수행
preds = model.predict(test_generator, verbose=1)
pred_labels = np.argmax(preds, axis=1)  # 예측 확률 → 클래스 index

9375/9375 [==============================] - 559s 59ms/step


In [27]:
# 4. 클래스 index → 클래스 이름 (LabelEncoder 사용해서 inverse_transform)
# 훈련 때 LabelEncoder를 le로 정의했음 (le.classes_ 순서로 인코딩)
pred_label_names = le.inverse_transform(pred_labels)

In [28]:
# 5. submission 파일에 결과 삽입
submission['label'] = pred_label_names

In [30]:
# 6. id에서 .png 제거
submission['id'] = submission['id'].str.replace('.png', '', regex=False)
print(submission['id'].head())

0    1
1    2
2    3
3    4
4    5
Name: id, dtype: object


In [31]:
# 7. 제출 파일 저장
submission.to_csv('submission_cifar10.csv', index=False)